In [1]:
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from utils import *
from sklearn.metrics import roc_auc_score

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
train_feats = pd.read_csv('../Data/train_features.csv')
oot1_feats = pd.read_csv('../Data/oot1_features.csv')
oot2_feats = pd.read_csv('../Data/oot2_features.csv')
features_dict = pickle.load(open('../Files/all_features.pkl', 'rb'))

In [3]:
all_features = list(set(sum(features_dict.values(), [])) - set(features_dict['identifier_features']))

### Descriptive statistics

In [4]:
null_df = pd.DataFrame({
    'null': train_feats[all_features].isna().sum(),
    'null_rate': train_feats[all_features].isna().sum()/train_feats.shape[0],
    'std': train_feats[all_features].std()
})
null_df.sort_values(by='null_rate', ascending=False)

/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/numpy/core/_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,null,null_rate,std
debt_to_ebitda,73229,0.154040,18265.573864
operating_margin,15583,0.032779,22546.372596
interest_coverage_proxy,15469,0.032540,32009.208549
cash_to_debt_st,952,0.002003,940.865557
current_assets_to_debt_st,952,0.002003,2358.670469
cash_flow_to_debt,189,0.000398,63.995016
bank_debt_concentration,189,0.000398,222.958660
st_debt_share,189,0.000398,0.320855
operating_cashflow_to_liabilities,93,0.000196,24.463025
equity_to_assets,0,0.000000,0.331097


### Univariate Analysis

In [5]:
def univariate_analysis(X, y, feature_categories):
    X = X.replace([np.inf, -np.inf], np.nan)
    y = pd.Series(np.asarray(y), index=X.index)

    results = []

    for category, features in feature_categories.items():
        print(category)
        for feature in tqdm(features):
            x = X[feature]
            valid = x.notna()

            if x[valid].nunique() < 2 or y[valid].nunique() < 2: continue

            auc = roc_auc_score(y[valid], x[valid])

            results.append({
                "category": category,
                "feature": feature,
                "auc": auc,
                "auc_strength": max(auc, 1 - auc),
                "missing_pct": 100 * (1 - valid.mean()),
                "n_observed": valid.sum()
            })
    return (
        pd.DataFrame(results)
        .sort_values(["category", "auc_strength"], ascending=[True, False])
        .reset_index(drop=True)
    )

In [6]:
feature_categories = {
    'size': ["log_assets"],
    'profitability': ["net_profit_to_assets","operating_margin","ebitda_to_assets","roa", "roe"],
    'liquidity': ["working_capital_to_assets","cash_to_assets","cash_to_debt_st","current_assets_to_asset_total","current_assets_to_debt_st"],
    'solvency': ["equity_to_assets","debt_to_assets","debt_st_to_assets","st_debt_share","bank_debt_concentration"],
    'debt_servicing': ["interest_coverage_proxy","cash_flow_to_debt","debt_to_ebitda"],
    'cash_flow': ["operating_cashflow_to_assets","operating_cashflow_to_liabilities","cashflow_profit_gap_to_assets"],
    'efficiency': ["sales_to_assets","receivables_to_assets","account_payables_to_assets"],
    'flags': features_dict['flag_features'],
    'categorical_feats': features_dict['categorical_features']
}
univariate_analysis_results = univariate_analysis(train_feats, train_feats['default'], feature_categories)

size


100%|██████████| 1/1 [00:00<00:00,  5.81it/s]


profitability


100%|██████████| 5/5 [00:00<00:00,  8.23it/s]


liquidity


100%|██████████| 5/5 [00:00<00:00,  8.02it/s]


... (8 outputs left out)

flags


100%|██████████| 10/10 [00:00<00:00, 17.12it/s]


categorical_feats


100%|██████████| 6/6 [00:00<00:00, 18.61it/s]


In [7]:
univariate_analysis_results

,category,feature,auc,auc_strength,missing_pct,n_observed
0,cash_flow,operating_cashflow_to_liabilities,0.258507,0.741493,0.019563,475296
1,cash_flow,operating_cashflow_to_assets,0.266384,0.733616,0.000000,475389
2,cash_flow,cashflow_profit_gap_to_assets,0.519475,0.519475,0.000000,475389
3,categorical_feats,legal_struct_SRU,0.530984,0.530984,0.000000,475389
4,categorical_feats,legal_struct_SPA,0.481655,0.518345,0.000000,475389
5,categorical_feats,legal_struct_SRL,0.491590,0.508410,0.000000,475389
6,categorical_feats,legal_struct_SAU,0.495825,0.504175,0.000000,475389
7,categorical_feats,legal_struct_SAA,0.499951,0.500049,0.000000,475389
8,categorical_feats,legal_struct_SRS,0.499995,0.500005,0.000000,475389
9,debt_servicing,cash_flow_to_debt,0.258406,0.741594,0.039757,475200


In [8]:
auc_selected_feats = list(univariate_analysis_results.query('auc_strength >= 0.55')['feature'])
len(auc_selected_feats)

24

### Correlation

In [9]:
def drop_corr_features(df, corr_feat_vars, threshold, univariate_analysis_results):
    corr_df = df[corr_feat_vars].corr()
    corr_pairs = corr_df.unstack()

    high_corr = corr_pairs[(abs(corr_pairs) > threshold) & (corr_pairs != 1.0)]
    high_corr = high_corr.drop_duplicates()
    print(high_corr)
    high_cor_vars = list(dict(high_corr).keys())
    print(high_cor_vars)

    # select the feature giving highest lift (precision per bin)
    def features_to_drop(corr_pairs):
        return univariate_analysis_results.query(f'feature in @corr_pairs').sort_values(by='auc_strength', ascending=True).iloc[0]['feature']
        
    corr_feats_to_drop = [features_to_drop(corr_pairs) for corr_pairs in high_cor_vars]
    return corr_feats_to_drop

In [10]:
high_corr_vars = drop_corr_features(train_feats, auc_selected_feats, 0.9, univariate_analysis_results)
high_corr_vars

ebitda_to_assets  roa               0.999831
equity_to_assets  debt_to_assets   -0.972868
dtype: float64
[('ebitda_to_assets', 'roa'), ('equity_to_assets', 'debt_to_assets')]


['roa', 'debt_to_assets']

### VIF (Impute, create flag and compute)

In [11]:
na_df = pd.DataFrame(train_feats.isna().sum())
na_df.columns = ['na_cnt']
cols_to_impute = list(na_df.query('na_cnt > 0').index)
cols_to_impute

['operating_margin',
 'cash_to_debt_st',
 'current_assets_to_debt_st',
 'st_debt_share',
 'bank_debt_concentration',
 'interest_coverage_proxy',
 'cash_flow_to_debt',
 'debt_to_ebitda',
 'operating_cashflow_to_liabilities']

In [12]:
feature_imputer = median_table_by_ateco_sector(train_feats, [c for c in sum(feature_categories.values(), []) if 'flag' not in c and 'legal_struct' not in c], asset_col='log_assets')
feature_imputer.to_csv('../Data/feature_imputer.csv', index=False)
feature_imputer = pd.read_csv('../Data/feature_imputer.csv')
feature_imputer

,ateco_sector,log_assets,net_profit_to_assets,operating_margin,ebitda_to_assets,roa,roe,working_capital_to_assets,cash_to_assets,cash_to_debt_st,...,bank_debt_concentration,interest_coverage_proxy,cash_flow_to_debt,debt_to_ebitda,operating_cashflow_to_assets,operating_cashflow_to_liabilities,cashflow_profit_gap_to_assets,sales_to_assets,receivables_to_assets,account_payables_to_assets
0,1.0,15.107622,-0.000085,0.001182,0.000775,0.010574,0.000604,0.000133,0.000297,0.001508,...,0.016467,0.021097,0.000655,1.068389,0.000372,0.000632,-0.000774,0.006147,0.004961,0.003406
1,2.0,14.801390,0.000015,0.002147,0.001157,0.048036,0.028304,0.001459,0.000464,0.001960,...,0.004984,0.070402,0.000515,1.068880,0.000373,0.000515,-0.000875,0.007681,0.009082,0.006273
2,3.0,14.869053,0.000015,0.002306,0.002701,0.083496,0.023340,0.004437,0.000321,0.000908,...,0.020726,0.072338,0.002672,0.693964,0.001493,0.002411,-0.001614,0.019705,0.009929,0.009151
3,5.0,16.948253,0.000262,0.002343,0.004679,0.267269,0.599554,0.008091,0.004141,0.005588,...,0.030181,0.131004,0.002988,0.529735,0.002725,0.002931,-0.003092,0.070490,0.021132,0.008044
4,6.0,16.029307,-0.000741,-0.015309,-0.000504,-0.112815,-0.315135,0.001865,0.000728,0.002112,...,0.000000,-0.524460,-0.000611,0.111748,-0.000165,-0.000393,-0.000221,0.004255,0.009489,0.002105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,94.0,15.278666,0.000932,0.009517,0.002729,0.186871,2.663926,0.001119,0.000210,0.000836,...,0.014340,0.109898,0.004356,0.616032,0.001648,0.004271,-0.000823,0.016085,0.012986,0.005475
80,95.0,14.747473,0.000388,0.002649,0.004756,0.295111,0.412947,0.008703,0.002726,0.005018,...,0.008301,0.180154,0.003195,0.614747,0.002246,0.002757,-0.001532,0.079322,0.034852,0.012306
81,96.0,15.056277,0.000115,0.002977,0.004341,0.157701,0.126351,-0.001651,0.000831,0.002625,...,0.019542,0.095277,0.004581,0.464172,0.002660,0.003926,-0.002338,0.034071,0.015008,0.007865
82,99.0,14.795285,-0.000026,0.004751,0.000778,0.047439,0.021854,0.004828,0.000119,0.001077,...,0.004188,0.052839,0.000040,1.753276,0.000026,0.000040,-0.000017,0.005239,0.004540,0.001901


In [13]:
def missing_value_imputation(df, imputation_table, cols_to_impute, asset_col='asst_tot'):
    df_copy = df.copy()
    for c in df.columns:
        if c in cols_to_impute: df[c] = pd.to_numeric(df[c], errors="coerce")

    # Create missingness flags before imputation
    flag_cols = {col: f"{col}_imputed_flag" for col in [asset_col, *cols_to_impute]}
    for col, flag_col in flag_cols.items(): df_copy[flag_col] = df_copy[col].isna().astype(int)

    imputation_table = imputation_table.set_index("ateco_sector")

    row_medians = imputation_table.reindex(df_copy["ateco_sector"]).set_axis(df_copy.index)
    row_medians = row_medians.fillna(imputation_table.loc["global"])
    assets = df_copy[asset_col].fillna(row_medians[asset_col])

    df_copy[cols_to_impute] = df_copy[cols_to_impute].fillna(row_medians[cols_to_impute].mul(assets, axis=0))

    return df_copy

In [14]:
train_feats_imputed = missing_value_imputation(train_feats, feature_imputer, cols_to_impute, asset_col='log_assets')
oot1_feats_imputed = missing_value_imputation(oot1_feats, feature_imputer, cols_to_impute, asset_col='log_assets')
oot2_feats_imputed = missing_value_imputation(oot2_feats, feature_imputer, cols_to_impute, asset_col='log_assets')

In [15]:
train_feats_imputed[auc_selected_feats].isna().sum()

operating_cashflow_to_liabilities    0
operating_cashflow_to_assets         0
cash_flow_to_debt                    0
interest_coverage_proxy              0
debt_to_ebitda                       0
receivables_to_assets                0
account_payables_to_assets           0
net_loss_flag                        0
operating_loss_flag                  0
nonpositive_ebitda_flag              0
negative_equity_flag                 0
cash_to_debt_st                      0
current_assets_to_debt_st            0
working_capital_to_assets            0
current_assets_to_asset_total        0
cash_to_assets                       0
net_profit_to_assets                 0
ebitda_to_assets                     0
roa                                  0
operating_margin                     0
equity_to_assets                     0
debt_to_assets                       0
debt_st_to_assets                    0
st_debt_share                        0
dtype: int64

In [16]:
[c for c in train_feats_imputed.columns if 'debt' in c and 'flag' in c]

['debt_cols_imputation_flag',
 'negative_debt_flag',
 'cash_to_debt_st_imputed_flag',
 'current_assets_to_debt_st_imputed_flag',
 'st_debt_share_imputed_flag',
 'bank_debt_concentration_imputed_flag',
 'cash_flow_to_debt_imputed_flag',
 'debt_to_ebitda_imputed_flag']

In [17]:
debt_missing_flag = ['cash_flow_to_debt_imputed_flag','debt_to_ebitda_imputed_flag','cash_to_debt_st_imputed_flag','current_assets_to_debt_st_imputed_flag','st_debt_share_imputed_flag']

train_feats_imputed['debt_missing_flag'] = train_feats_imputed[debt_missing_flag].eq(1).any(axis=1).astype(int)
oot1_feats_imputed['debt_missing_flag'] = oot1_feats_imputed[debt_missing_flag].eq(1).any(axis=1).astype(int)
oot2_feats_imputed['debt_missing_flag'] = oot2_feats_imputed[debt_missing_flag].eq(1).any(axis=1).astype(int)
train_feats_imputed.drop(columns=debt_missing_flag, inplace=True)
oot1_feats_imputed.drop(columns=debt_missing_flag, inplace=True)
oot2_feats_imputed.drop(columns=debt_missing_flag, inplace=True)

In [19]:
train_feats_imputed.to_csv('../Data/train_features_imputed.csv')
oot1_feats_imputed.to_csv('../Data/oot1_features_imputed.csv')
oot2_feats_imputed.to_csv('../Data/oot2_features_imputed.csv')

In [20]:
oot2_feats_imputed.columns

Index(['id', 'stmt_date', 'data_type', 'default', 'ateco_sector',
       'debt_cols_imputation_flag', 'asset_cols_imputation_flag',
       'income_statement_cols_imputation_flag', 'log_assets',
       'net_profit_to_assets', 'operating_margin', 'ebitda_to_assets', 'roa',
       'roe', 'working_capital_to_assets', 'cash_to_assets', 'cash_to_debt_st',
       'current_assets_to_asset_total', 'current_assets_to_debt_st',
       'equity_to_assets', 'debt_to_assets', 'debt_st_to_assets',
       'st_debt_share', 'bank_debt_concentration', 'interest_coverage_proxy',
       'cash_flow_to_debt', 'debt_to_ebitda', 'operating_cashflow_to_assets',
       'operating_cashflow_to_liabilities', 'cashflow_profit_gap_to_assets',
       'sales_to_assets', 'receivables_to_assets',
       'account_payables_to_assets', 'negative_equity_flag',
       'negative_debt_flag', 'operating_loss_flag', 'net_loss_flag',
       'nonpositive_revenue_flag', 'zero_financing_expense_flag',
       'nonpositive_ebitda_flag',

In [21]:
flag_cols = [c for c in train_feats_imputed.columns if 'flag' in c]
flag_corr = train_feats_imputed[flag_cols].corr()
flag_cols_to_drop = []

# Find nearly perfectly correlated flags
for i, col1 in enumerate(flag_cols):
    for col2 in flag_cols[i+1:]:
        corr = flag_corr.loc[col1, col2]

        if pd.notna(corr) and abs(corr) > 0.999:
            print(col1, "<-->", col2, "Correlation:", corr)
            flag_cols_to_drop.append(col1)

nonpositive_revenue_flag <--> operating_margin_imputed_flag Correlation: 1.0


In [22]:
final_selected_feats = [c for c in auc_selected_feats if c not in high_corr_vars + flag_cols_to_drop]

In [23]:
def compute_vif(df, cols):
    X_vif = sm.add_constant(df[cols])
    vif = pd.DataFrame({
        "feature": X_vif.columns,
        "VIF": [
            variance_inflation_factor(X_vif.to_numpy(), i) for i in range(X_vif.shape[1])
        ],
    })

    return vif.sort_values("VIF", ascending=False)

In [24]:
vif_df = compute_vif(train_feats_imputed, final_selected_feats)
vif_df

/var/folders/22/3z4yr_bd5ds4d1g7ph9nprs80000gn/T/ipykernel_52345/356350387.py:6: UserWarning: The design matrix is poorly conditioned (condition number=2.05e+05). VIF calculations may be numerically unstable.
  variance_inflation_factor(X_vif.to_numpy(), i) for i in range(X_vif.shape[1])


,feature,VIF
14,working_capital_to_assets,4.851626e+09
15,current_assets_to_asset_total,4.258441e+09
21,debt_st_to_assets,4.199274e+09
22,st_debt_share,3.651122e+00
20,equity_to_assets,3.439333e+00
9,operating_loss_flag,3.308201e+00
10,nonpositive_ebitda_flag,2.857764e+00
6,receivables_to_assets,1.740137e+00
7,account_payables_to_assets,1.602654e+00
8,net_loss_flag,1.587438e+00


In [26]:
vif_df_2 = compute_vif(train_feats_imputed, [c for c in final_selected_feats if c in list(vif_df.query('VIF <= 5')['feature'])])
vif_df_2

,feature,VIF
9,operating_loss_flag,3.240311
10,nonpositive_ebitda_flag,2.840969
8,net_loss_flag,1.578890
7,account_payables_to_assets,1.533221
6,receivables_to_assets,1.522020
19,st_debt_share,1.423533
1,operating_cashflow_to_liabilities,1.385811
12,cash_to_debt_st,1.332654
18,equity_to_assets,1.276297
13,current_assets_to_debt_st,1.271974


In [27]:
vif_df.to_csv('../Files/VIF.csv', index=False)
vif_df_2.to_csv('../Files/VIF_2.csv', index=False)

### Save the final list of features

In [28]:
pickle.dump({'final_selected_features_before_vif': final_selected_feats}, open('../Files/selected_features_before_vif.csv', 'wb'))

In [29]:
pickle.load(open('../Files/selected_features_before_vif.csv', 'rb'))

{'final_selected_features_before_vif': ['operating_cashflow_to_liabilities',
  'operating_cashflow_to_assets',
  'cash_flow_to_debt',
  'interest_coverage_proxy',
  'debt_to_ebitda',
  'receivables_to_assets',
  'account_payables_to_assets',
  'net_loss_flag',
  'operating_loss_flag',
  'nonpositive_ebitda_flag',
  'negative_equity_flag',
  'cash_to_debt_st',
  'current_assets_to_debt_st',
  'working_capital_to_assets',
  'current_assets_to_asset_total',
  'cash_to_assets',
  'net_profit_to_assets',
  'ebitda_to_assets',
  'operating_margin',
  'equity_to_assets',
  'debt_st_to_assets',
  'st_debt_share']}